<a href="https://colab.research.google.com/github/Datdeptrai912005/DemoGit/blob/main/MLLM_Qwen_Trend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

train_dataset = load_dataset("json", data_files="/content/drive/MyDrive/train_data.jsonl", split="train")
test_dataset = load_dataset("json", data_files="/content/drive/MyDrive/test_data.jsonl", split="train")
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
test_dataset = test_dataset.map(formatting_prompts_func, batched=True)

print(f"📚 Đã nạp {len(train_dataset)} mẫu để Dạy (Train)")
print(f"📝 Đã nạp {len(test_dataset)} mẫu để Kiểm tra (Test)")

In [ ]:
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Tải mô hình thành công ")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 20,
        eval_strategy = "steps",
        eval_steps = 100,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)
trainer_stats = trainer.train()

In [ ]:
save_path = "/content/drive/MyDrive/MLLA /Qwen_FakeNews_Model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Đã lưu thành công trí tuệ của mô hình tại: {save_path}")

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from tqdm import tqdm

print("Đang tải lại mô hình Qwen từ Drive...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/MLLA /Qwen_FakeNews_Model",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

print("Đang tải dữ liệu test")
test_dataset = load_dataset("json", data_files="/content/drive/MyDrive/test_data.jsonl", split="train")

y_true = []
y_pred = []

print(f"Bắt đầu cho Qwen làm đề thi gồm {len(test_dataset)} câu (Sẽ mất vài phút)...")
for item in tqdm(test_dataset):
    # Lấy đáp án thật từ file json
    true_label = next(msg["content"] for msg in item["messages"] if msg["role"] == "assistant")
    y_true.append(int(true_label))

    # Lấy nội dung tin tức cần kiểm tra
    user_text = next(msg["content"] for msg in item["messages"] if msg["role"] == "user")

    # Thiết lập ngữ cảnh cho AI
    messages = [
        {"role": "system", "content": "Phân tích tin giả."},
        {"role": "user", "content": user_text}
    ]

    # Đưa vào mô hình để suy luận
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=10, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.batch_decode(outputs)[0]

    # Cắt lấy phần câu trả lời của AI
    pred_str = response.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    if "1" in pred_str:
        y_pred.append(1)
    else:
        y_pred.append(0)

print("Đã phân tích xong biểu đồ đang được hoàn thiện ")
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Tin thật (0)', 'Tin giả (1)'],
            yticklabels=['Tin thật (0)', 'Tin giả (1)'])

plt.title('Sơ đồ Phân tích lỗi của mô hình - Qwen2.5', fontsize=15, fontweight='bold')
plt.xlabel('AI Dự Đoán', fontsize=12)
plt.ylabel('Đáp Án Thực Tế', fontsize=12)

plt.show()